In [19]:
import sys

sys.path.append("../src/")
import mlflow
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mlflow.tracking import MlflowClient
from scipy.stats import pearsonr

client = MlflowClient(tracking_uri="http://localhost:5000")
experiment_id_mapping = experiment_id_mapping = {
    "tiny-imagenet-vit": "727",
}


def get_results(experiment_name):
    experiment_id = experiment_id_mapping[experiment_name]
    methods = [
        # "original",
        "retrained",
        "finetune",
        "neggrad",
        "relabel",
        "badT",
        "scrub",
        "ssd",
        "unsir",
        "salun_relabel",
        "our",
        "salun_lotus",
    ]
    # methods.extend(our_method_names)

    metrics = [
        "mia",
        "acc_forget",
        "acc_retain",
        "acc_test",
        "acc_val",
        "js",
        "t",
        "js_proxy",
    ]
    runs = client.search_runs(experiment_id)
    df = pd.DataFrame(
        [
            {k: v for k, v in run.data.metrics.items() if k in metrics}
            for run in runs
            if run.data.params.get("method") in methods
        ]
    )

    # List of columns to divide
    columns_to_divide = ["mia", "acc_forget", "acc_retain", "acc_test", "acc_val"]

    # Perform the division for the specified columns
    df[columns_to_divide] = df[columns_to_divide] / 100

    df["method"] = [
        run.data.params.get("method")
        for run in runs
        if run.data.params.get("method") in methods
    ]

    df["seed"] = [
        run.data.params.get("seed")
        for run in runs
        if run.data.params.get("method") in methods
    ]

    df.set_index(["method", "seed"], inplace=True)

    grouped_df = df.groupby("method").aggregate(["mean", "std"])
    grouped_df["js"] = grouped_df["js"].apply(lambda x: x * 1e4)
    grouped_df = grouped_df.round(2)

    gap_metrics = ["mia", "acc_forget", "acc_retain", "acc_test", "acc_val"]
    for method in methods:
        for metric in gap_metrics:
            grouped_df.loc[method, f"{metric}_gap"] = abs(
                grouped_df.loc[method, (metric, "mean")]
                - grouped_df.loc["retrained", (metric, "mean")]
            )
    grouped_df["avg_gap"] = (
        grouped_df[
            [
                "mia_gap",
                "acc_retain_gap",
                "acc_forget_gap",
                "acc_test_gap",
            ]
        ]
        .mean(axis=1)
        .round(4)
    )

    main_df = grouped_df.loc[
        [m for m in methods if m in grouped_df.index],
        ["avg_gap", "js", 't', 'js_proxy'],
    ]
    # ].sort_values(by=("avg_gap", ""), ascending=True)
    # ].sort_values(by=("js", "mean"), ascending=True)
    methods_for_appendix = [m for m in methods if m in grouped_df.index]

    appendix_df = grouped_df.loc[
        methods_for_appendix,
        [
            "avg_gap",
            "mia_gap",
            "acc_forget_gap",
            "acc_retain_gap",
            "acc_test_gap",
            "mia",
            "acc_forget",
            "acc_retain",
            "acc_test",
        ],
    ]
    # appendix_df = appendix_df.sort_values(by=("avg_gap", ""), ascending=True)

    print(experiment_name)

    display(main_df)
    display(appendix_df)

    return main_df, appendix_df

In [27]:
vit_tiny_imagenet, _ = get_results("tiny-imagenet-vit")

tiny-imagenet-vit


avg_gap    js             t       js_proxy     
                       mean   std    mean   std     mean  std
method                                                       
retrained      0.0000  0.00  0.00  228.90  6.49      0.0  0.0
finetune       0.0175  0.05  0.00   22.64  0.02      0.0  0.0
neggrad        0.0400  0.10  0.00   25.20  0.02      0.0  0.0
relabel        0.2925  0.64  1.03   25.19  0.02      0.0  0.0
badT           0.0775  0.18  0.01   16.91  0.05      0.0  0.0
scrub          0.0225  0.04  0.00   33.25  0.01      0.0  0.0
ssd            0.0225  0.04  0.00   27.27  0.06      0.0  0.0
unsir          0.0225  0.06  0.00   21.17  0.22      0.0  0.0
salun_relabel  0.0925  0.25  0.59   76.97  1.72      0.0  0.0
our            0.0150  0.03  0.00   13.41  0.04      0.0  0.0
salun_lotus       NaN   NaN   NaN     NaN   NaN      NaN  NaN

avg_gap mia_gap acc_forget_gap acc_retain_gap acc_test_gap  \
                                                                           
method                                                                     
retrained      0.0000    0.00           0.00           0.00         0.00   
finetune       0.0175    0.02           0.03           0.02         0.00   
neggrad        0.0400    0.07           0.07           0.02         0.00   
relabel        0.2925    0.26           0.29           0.32         0.30   
badT           0.0775    0.09           0.06           0.09         0.07   
scrub          0.0225    0.03           0.06           0.00         0.00   
ssd            0.0225    0.03           0.06           0.00         0.00   
unsir          0.0225    0.04           0.02           0.02         0.01   
salun_relabel  0.0925    0.09           0.08           0.10         0.10   
our            0.0150    0.00           0.06           0.00         0.00   
salun_lotus       NaN     NaN            NaN            NaN          NaN   

                mia       acc_forget       acc_retain       acc_test        
               mean   std       mean   std       mean   std     mean   std  
method                                                                      
retrained      0.76  0.00       0.90  0.00       0.96  0.00     0.90  0.00  
finetune       0.78  0.00       0.93  0.00       0.98  0.00     0.90  0.00  
neggrad        0.83  0.00       0.97  0.00       0.98  0.00     0.90  0.00  
relabel        0.50  0.43       0.61  0.52       0.64  0.55     0.60  0.51  
badT           0.67  0.00       0.84  0.01       0.87  0.01     0.83  0.01  
scrub          0.79  0.00       0.96  0.00       0.96  0.00     0.90  0.00  
ssd            0.79  0.00       0.96  0.00       0.96  0.00     0.90  0.00  
unsir          0.80  0.00       0.92  0.00       0.94  0.00     0.89  0.00  
salun_relabel  0.67  0.25       0.82  0.30       0.86  0.32     0.80  0.30  
our            0.76  0.00       0.96  0.00       0.96  0.00     0.90  0.00  
salun_lotus     NaN   NaN        NaN   NaN        NaN   NaN      NaN   NaN